*❗❗ Before you run this lab, go to Runtime -> Change Runtime Type -> Choose: T4 GPU*

# Module 4 Lab: Investigating the Image Retrieval Engine
**AIML 2013 — Computer Vision**

*One image dataset. One vector database. What does "similar" actually mean?*

---

**This is the standalone CV lab for Module 4.** You will work in one notebook, give one 3–5 minute demo, and submit your GitHub repo link to Canvas.


## How This Lab Works

This lab is different from previous labs.

In Modules 2 and 3, you built pipelines from scratch — HOG descriptors, CNN feature extraction, cosine similarity search. You know how to construct these systems. This week, construction isn't the point.

Part 1 gives you a working image retrieval system: dataset loading, MobileNetV2 feature extraction, ChromaDB vector storage, and similarity search with visualization. The code is all visible — read it as you run it — but you don't need to write it.

Your job is to **run the system, then investigate it.** Part 2 contains four experiments. Each one asks a question about how the image vector space works, gives you a procedure to answer it, and asks you to record what you found. The experiments build on each other. By the end, you'll understand things about embeddings that you can't learn by building a pipeline — things you can only learn by poking at one and watching what happens.

**For each experiment:** write your code, run it, and then write a markdown cell explaining what you observed and what it means. The markdown cells are not filler. They're the core of your demo.


---
# Part 1: The System

Run these cells in order. Read the code as you go. By the end of Part 1, you'll have a working image retrieval system backed by ChromaDB.

**Do not modify Part 1 unless something breaks.** The experiments in Part 2 depend on the variable names and data structures created here.


In [ ]:
# Force Text Wrapping for all outputs
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap !important;
        word-break: break-word !important;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)


### Cell 1: Setup and Dependencies
Installs all libraries and imports everything for the image pipeline.

You can disregard dependency errors here if you get an "All imports succeeded" message.


In [ ]:
# Cell 1: Setup and dependencies
!pip install -q chromadb sentence-transformers google-genai tensorflow scikit-image einops

import numpy as np
import matplotlib.pyplot as plt
import chromadb
from google import genai
import tensorflow as tf
from tensorflow.keras.applications import mobilenet_v2
from PIL import Image
from skimage import transform
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from sentence_transformers import SentenceTransformer
import math
import base64

print(f"chromadb {chromadb.__version__}")
print("All imports succeeded.")


### Cell 2: Configure Gemini API

You will need to grant access to your Google API key when prompted.

If you get an authentication error, check that the secret is named exactly `GEMINI_API_KEY` and that notebook access is toggled on.


In [ ]:
# Cell 2: Configure the Gemini API
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Say hello in exactly five words.'
)
print(response.text)


### Cell 3: Load and Preprocess Images

Choose one of two image datasets by setting `DATASET` in the cell below.

- **`"flowers"`** — Oxford/TF Flowers. Five species of flowers (daisy, dandelion, roses, sunflowers, tulips). Natural photos at 300–500px. Fine-grained: the classes look similar, so retrieval has to work harder to tell them apart.
- **`"imagenette"`** — A 10-class subset of ImageNet (tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute). Visually diverse: good for testing whether the model separates distinct objects.

Cell 3b resizes all images to 224×224 for MobileNetV2 and applies the model’s preprocessing function. It also generates an index sheet (`image_index_sheet.jpg`) you’ll reference in Experiment 1.

**Caching:** Images and preprocessed arrays are cached to Google Drive. Subsequent runs load from cache.


In [ ]:
# Cell 3: Load and preprocess images (with Google Drive caching)
import os, pathlib, urllib.request, tarfile
import numpy as np
from google.colab import drive
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

# Mount Drive and setup cache folder
drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/AIML_Lab_Cache'
os.makedirs(CACHE_DIR, exist_ok=True)

DATASET = "flowers"  # <- change to "imagenette" for the 10-class object dataset

cached_images_path = os.path.join(CACHE_DIR, f"{DATASET}_images.npz")

if os.path.exists(cached_images_path):
    print(f"Loading cached {DATASET} images from Google Drive...")
    data = np.load(cached_images_path, allow_pickle=True)
    images_raw = list(data['images_raw'])
    labels = list(data['labels'])
else:
    print(f"No cached images found. Downloading and processing {DATASET} dataset...")
    if DATASET == "flowers":
        data_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
        extracted_path = tf.keras.utils.get_file("flower_photos", origin=data_url, untar=True)
        data_dir = pathlib.Path(extracted_path)
        if (data_dir / "flower_photos").exists():
            data_dir = data_dir / "flower_photos"
        license_file = data_dir / "LICENSE.txt"
        if license_file.exists():
            license_file.unlink()

    elif DATASET == "imagenette":
        data_url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
        archive_path = os.path.join("/content", "imagenette2-160.tgz")
        if not os.path.exists("/content/imagenette2-160"):
            print("Downloading Imagenette (~100MB)...")
            urllib.request.urlretrieve(data_url, archive_path)
            with tarfile.open(archive_path) as tar:
                tar.extractall("/content")
        data_dir = pathlib.Path("/content/imagenette2-160/train")
        imagenette_labels = {
            "n01440764": "tench", "n02102040": "springer",
            "n02979186": "cassette player", "n03000684": "chain saw",
            "n03028079": "church", "n03394916": "French horn",
            "n03417042": "garbage truck", "n03425413": "gas pump",
            "n03445777": "golf ball", "n03888257": "parachute"
        }
    else:
        raise ValueError(f"Unknown dataset: {DATASET}. Use 'flowers' or 'imagenette'.")

    N_PER_CLASS = 20
    OFFSET = 20

    images_raw = []
    labels = []
    class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])

    for class_dir in class_dirs:
        class_name = class_dir.name
        if DATASET == "imagenette":
            class_name = imagenette_labels.get(class_name, class_name)

        image_files = sorted(class_dir.glob("*.jpg")) + sorted(class_dir.glob("*.JPEG"))

        if len(image_files) > OFFSET:
            image_files = image_files[OFFSET : OFFSET + N_PER_CLASS]
        else:
            image_files = image_files[:N_PER_CLASS]

        for img_path in image_files:
            try:
                img = Image.open(img_path).convert("RGB")
                img_array = np.array(img)
                if img_array.ndim == 3 and img_array.shape[2] == 3:
                    images_raw.append(img_array)
                    labels.append(class_name)
            except Exception:
                continue

    print("Saving raw images to Google Drive cache...")
    np.savez(cached_images_path, images_raw=np.array(images_raw, dtype=object), labels=np.array(labels))

print(f"Dataset: {DATASET}")
print(f"Images loaded: {len(images_raw)}")
print(f"Classes: {sorted(set(labels))}")
print(f"Per class: {dict(Counter(labels))}")

# Show a sample
sample_classes = sorted(set(labels))[:4]
fig, axes = plt.subplots(len(sample_classes), 4, figsize=(12, 3 * len(sample_classes)))
for row, cls in enumerate(sample_classes):
    cls_imgs = [img for img, l in zip(images_raw, labels) if l == cls][:4]
    for col, img in enumerate(cls_imgs):
        axes[row][col].imshow(img)
        axes[row][col].set_title(cls if col == 0 else "", fontsize=10)
        axes[row][col].axis('off')
plt.suptitle(f"{DATASET}: sample images", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 3b: Resize and preprocess for MobileNetV2 (with caching)
import os
import numpy as np
from PIL import Image
from tensorflow.keras.applications import mobilenet_v2

cached_resized_path = os.path.join(CACHE_DIR, f"{DATASET}_images_resized.npy")
cached_preprocessed_path = os.path.join(CACHE_DIR, f"{DATASET}_images_preprocessed.npy")

if os.path.exists(cached_resized_path) and os.path.exists(cached_preprocessed_path):
    print(f"Loading cached resized and preprocessed {DATASET} images...")
    images_resized = np.load(cached_resized_path)
    images_preprocessed = np.load(cached_preprocessed_path)
else:
    print("Resizing and preprocessing images...")
    images_resized = np.array([
        np.array(Image.fromarray(img).resize((224, 224)))
        for img in images_raw
    ])

    images_preprocessed = mobilenet_v2.preprocess_input(
        images_resized.astype(np.float32).copy()
    )

    print("Saving intermediate resizings to Google Drive cache...")
    np.save(cached_resized_path, images_resized)
    np.save(cached_preprocessed_path, images_preprocessed)

print(f"Resized shape: {images_resized.shape}")
print(f"Preprocessed range: [{images_preprocessed.min():.1f}, {images_preprocessed.max():.1f}]")
print(f"Ready for MobileNetV2.")


### Cell 4: Extract Image Embeddings

Loads MobileNetV2 as a feature extractor (no classification head, global average pooling) and produces a 1,280-dimensional embedding per image.

**Caching:** Image embeddings are cached to Google Drive.


In [ ]:
# Cell 4: Extract image embeddings (with Google Drive caching)
import os
import numpy as np
import tensorflow as tf

CACHE_DIR = '/content/drive/MyDrive/AIML_Lab_Cache'
image_emb_path = os.path.join(CACHE_DIR, f'{DATASET}_image_embeddings.npy')

base_model = tf.keras.applications.MobileNetV2(
    include_top=False, weights='imagenet', pooling='avg'
)

if os.path.exists(image_emb_path):
    print(f"Loading {DATASET} image embeddings from Google Drive cache...")
    image_embeddings = np.load(image_emb_path)
else:
    print("Computing image embeddings...")
    image_embeddings = base_model.predict(images_preprocessed, verbose=1)
    np.save(image_emb_path, image_embeddings)
    print("Saved image embeddings to Google Drive.")

print(f"Image embedding shape: {image_embeddings.shape}")
print(f"Dimensionality: {image_embeddings.shape[1]}")
density = np.count_nonzero(image_embeddings) / image_embeddings.size * 100
print(f"Density: {density:.1f}%")


### Cell 5: Build ChromaDB Image Collection

ChromaDB is a vector database. Traditional databases store rows and match exact keywords. Vector databases store embedding vectors and match by similarity — give me the 5 nearest vectors to this query. SQL matches words. Vectors match meaning.

Cell 5b lets you peek inside the database — what a single record looks like, how the embedding vector is stored alongside the image label.


In [ ]:
# Cell 5: Create ChromaDB image collection
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("image_embeddings")
except Exception:
    pass

image_collection = chroma_client.create_collection(name="image_embeddings")
image_collection.add(
    ids=[f"img_{i}" for i in range(len(labels))],
    embeddings=image_embeddings.tolist(),
    documents=labels,
    metadatas=[{"label": labels[i], "index": i} for i in range(len(labels))]
)

print(f"Image collection: {image_collection.count()} items, {image_embeddings.shape[1]}d")


In [ ]:
# Cell 5b: Peek inside the vector database
import random

random_img_idx = random.randint(0, len(labels) - 1)

sample_img = image_collection.get(ids=[f"img_{random_img_idx}"], include=["embeddings", "documents", "metadatas"])

print("=== One record from the image collection ===")
print(f"  ID:        {sample_img['ids'][0]}")
print(f"  Document:  {sample_img['documents'][0]}")
print(f"  Metadata:  {sample_img['metadatas'][0]}")
img_emb = sample_img['embeddings'][0]
print(f"  Embedding (first 40 dims): {[round(float(x), 4) for x in img_emb[:40]]}")
print(f"  ... ({len(img_emb)} dimensions total)")

print()
print("Each image is stored as a 1,280-dimensional vector.")
print("To find similar images, ChromaDB computes the distance between")
print("the query vector and every stored vector, then returns the closest ones.")


### Cell 6: Search and Visualization Functions
Defines image search and display functions. Also generates an index sheet of all images with their indices.


In [ ]:
# Cell 6: Search and visualization functions
from IPython.display import HTML, display

def search_images_by_image(query_idx, n=5):
    query_emb = image_embeddings[query_idx].tolist()
    results = image_collection.query(query_embeddings=[query_emb], n_results=n + 1)
    filtered = {"ids": [[]], "documents": [[]], "distances": [[]], "metadatas": [[]]}
    for j, mid in enumerate(results["metadatas"][0]):
        if mid["index"] != query_idx:
            filtered["ids"][0].append(results["ids"][0][j])
            filtered["documents"][0].append(results["documents"][0][j])
            filtered["distances"][0].append(results["distances"][0][j])
            filtered["metadatas"][0].append(results["metadatas"][0][j])
    for key in filtered:
        filtered[key][0] = filtered[key][0][:n]
    return filtered

def show_image_retrieval(query_idx, n=3):
    results = search_images_by_image(query_idx, n=n)
    fig, axes = plt.subplots(1, n + 1, figsize=(3 * (n + 1), 3))

    axes[0].imshow(images_resized[query_idx])
    axes[0].set_title(f"Query: {labels[query_idx]}\n(idx: {query_idx})", fontweight='bold')
    axes[0].axis('off')

    for i, meta in enumerate(results["metadatas"][0]):
        idx = meta["index"]
        dist = results["distances"][0][i]
        match = "+" if labels[idx] == labels[query_idx] else "-"
        axes[i + 1].imshow(images_resized[idx])
        axes[i + 1].set_title(f"{labels[idx]} {match}\n{1 - dist:.3f} sim\n(idx: {idx})")
        axes[i + 1].axis('off')

    plt.tight_layout()
    plt.show()

def create_index_sheet():
    n_images = len(images_resized)
    cols = 10
    rows = math.ceil(n_images / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    axes = axes.flatten()

    for idx in range(n_images):
        axes[idx].imshow(images_resized[idx])
        axes[idx].set_title(f"idx: {idx}", fontsize=9)
        axes[idx].axis('off')

    for idx in range(n_images, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    sheet_path = "/content/image_index_sheet.jpg"
    plt.savefig(sheet_path, dpi=150)
    plt.show()

    with open(sheet_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    html_link = f'<div style="margin-top: 10px;"><a href="data:image/jpeg;base64,{b64}" download="image_index_sheet.jpg" style="font-size: 16px; font-weight: bold; color: #1a73e8; text-decoration: none; padding: 8px 12px; border: 1px solid #1a73e8; border-radius: 4px; display: inline-block;">Download Full Index Sheet</a></div>'
    display(HTML(html_link))
    print(f"\nIndex sheet saved to {sheet_path}")

create_index_sheet()
show_image_retrieval(0, n=3)


---
**Part 1 checkpoint.** You should now have a working image retrieval system backed by ChromaDB. If anything above threw an error, fix it before continuing — every experiment in Part 2 depends on these variables and functions.

---


# Part 2: Experiments

Four experiments, each asking a different question about how the image vector space works. Run the code, record what you find, and write the markdown reflection for each one. The reflections are the substance of your demo.


## Experiment 1: The Similarity Game

**The question:** Does MobileNetV2's notion of "similar" match yours? When you look at an image and predict which other images the model will rank as most similar, are you right?

**The procedure:**

1. The cell below displays your image index sheet — a grid of all images with their indices and class labels.
2. Pick a query image. Before running retrieval, write down which image you predict will be the model's #1 match and why.
3. Run retrieval. Compare the model's results to your prediction.
4. Repeat for at least five images from different classes.


In [ ]:
# Experiment 1: The Similarity Game — Images
all_classes = sorted(set(labels))
print(f"Available classes: {all_classes}\n")

from IPython.display import Image as IPImage, display
print("Review the index sheet below (or open /content/image_index_sheet.jpg) to pick your query images.")
display(IPImage("/content/image_index_sheet.jpg"))


In [ ]:
# Run retrieval for your chosen query image
# Change QUERY_IDX each time. Write your prediction BEFORE running.

QUERY_IDX = 12
MY_PREDICTION = "I think idx=4 will be #1 because..."

if 'image_attempts' not in dir():
    image_attempts = []

show_image_retrieval(QUERY_IDX, n=4)

results = search_images_by_image(QUERY_IDX, n=1)
actual_top = results["metadatas"][0][0]["index"]

correct = "✓ Correct!" if str(actual_top) in MY_PREDICTION else "✗ Model disagreed."
image_attempts.append({
    "query_idx": QUERY_IDX,
    "query_label": labels[QUERY_IDX],
    "prediction": MY_PREDICTION,
    "actual_top": actual_top,
    "actual_label": labels[actual_top]
})

print(f"\nYour prediction: {MY_PREDICTION}")
print(f"Model's #1:      idx={actual_top} ({labels[actual_top]})")
print(f"Result:           {correct}")
print(f"\nAttempts so far: {len(image_attempts)}")
for a in image_attempts:
    match = "+" if a["query_label"] == a["actual_label"] else "-"
    print(f"  Query idx={a['query_idx']} ({a['query_label']}) -> Model picked idx={a['actual_top']} ({a['actual_label']}) {match}")


**✍️ Experiment 1 Reflection** (write your answers here)

1. How many of your predictions matched the model's top result? When you were wrong, what did the model pick instead?
2. What visual features seem to drive the model's similarity judgments? Color? Shape? Texture? Object identity?


## Experiment 2: Embedding Surgery

**The question:** Embedding vectors aren't just opaque numbers — they have geometric structure. If you average two embeddings, does the midpoint land somewhere meaningful?

**The procedure:**

1. Pick two images from different classes. Average their embeddings. Retrieve the closest image to the midpoint.
2. Try it with three images from three different classes.
3. Try averaging two images from the *same* class. Does the midpoint retrieve something "more typical"?
4. Try subtracting: take image A's embedding, subtract image B's embedding, and retrieve the nearest image.


In [ ]:
# Experiment 2: Image embedding surgery
import random

all_classes = sorted(set(labels))
print(f"Available classes: {all_classes}\n")

for cls in all_classes:
    cls_indices = [j for j, l in enumerate(labels) if l == cls]
    print(f"  {cls}: indices {cls_indices[0]}–{cls_indices[-1]} ({len(cls_indices)} images)")

###################################################
# Change these ⬇️ to pick your images
CLASS_A = "daisy"
CLASS_B = "sunflowers"
###################################################

IMG_A = random.choice([i for i, l in enumerate(labels) if l == CLASS_A])
IMG_B = random.choice([i for i, l in enumerate(labels) if l == CLASS_B])

print(f"\nImage A: idx={IMG_A}, class={labels[IMG_A]}")
print(f"Image B: idx={IMG_B}, class={labels[IMG_B]}")

img_midpoint = (image_embeddings[IMG_A] + image_embeddings[IMG_B]) / 2.0

# Fetch extra results so we can filter out the source images
mid_img_results = image_collection.query(
    query_embeddings=[img_midpoint.tolist()],
    n_results=10
)

# Filter out source images A and B
filtered_metas = []
filtered_dists = []
for meta, dist in zip(mid_img_results['metadatas'][0], mid_img_results['distances'][0]):
    if meta['index'] in [IMG_A, IMG_B]:
        continue
    filtered_metas.append(meta)
    filtered_dists.append(dist)
    if len(filtered_metas) >= 5:
        break

fig, axes = plt.subplots(1, 2 + len(filtered_metas), figsize=(3 * (2 + len(filtered_metas)), 3))

axes[0].imshow(images_resized[IMG_A])
axes[0].set_title(f"Image A\n{labels[IMG_A]} (idx={IMG_A})", fontweight='bold', fontsize=9)
axes[0].axis('off')

axes[1].imshow(images_resized[IMG_B])
axes[1].set_title(f"Image B\n{labels[IMG_B]} (idx={IMG_B})", fontweight='bold', fontsize=9)
axes[1].axis('off')

for j, (meta, dist) in enumerate(zip(filtered_metas, filtered_dists)):
    idx = meta['index']
    axes[j + 2].imshow(images_resized[idx])
    axes[j + 2].set_title(f"{labels[idx]}\n{1-dist:.3f} sim\n(idx={idx})", fontsize=9)
    axes[j + 2].axis('off')

plt.suptitle(f"Nearest to Midpoint of {labels[IMG_A]} + {labels[IMG_B]} (excluding A & B)", fontsize=12)
plt.tight_layout()
plt.show()


**✍️ Experiment 2 Reflection** (write your answers here)

1. For the cross-class midpoint: did the retrieved images relate to both source classes, to one of them, or to something else entirely?
2. What happened when you averaged two images from the same class? Did the midpoint retrieve something "more typical"?


## Experiment 3: The Caption Bridge

**The question:** Your ChromaDB collection indexes images by visual similarity. But what if you wanted to search by text? "Show me a red flower" or "find the church." The image embeddings (1,280d from MobileNetV2) and text embeddings (768d from a sentence transformer) live in different vector spaces.

The workaround: caption every image with Gemini Vision, embed the captions as text, and build a second ChromaDB collection.

**The procedure:**

1. Run the setup cell to caption all images and build a text index.
2. Write five text queries describing images you know are in your dataset.
3. For each query, compare what you expected to find with what the caption-based search returned.
4. Find a case where the bridge works well and a case where it breaks.


In [ ]:
# Experiment 3 setup: Caption all images and build a text index
import io
import base64
import time
from PIL import Image as PILImage

text_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)

caption_cache_path = os.path.join(CACHE_DIR, f'{DATASET}_captions.npz')

if os.path.exists(caption_cache_path):
    print("Loading cached captions from Google Drive...")
    cap_data = np.load(caption_cache_path, allow_pickle=True)
    captions = list(cap_data['captions'])
else:
    print(f"Captioning {len(images_resized)} images with Gemini Vision...")
    print("This takes 1-2 minutes. Each image gets a one-sentence caption.\n")
    captions = []
    for i, img in enumerate(images_resized):
        pil_img = PILImage.fromarray(img.astype(np.uint8))
        buf = io.BytesIO()
        pil_img.save(buf, format='JPEG')
        b64 = base64.b64encode(buf.getvalue()).decode('utf-8')

        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[
                {"text": "Describe this image in one detailed sentence. Be specific about what you see — colors, objects, actions, context."},
                {"inline_data": {"mime_type": "image/jpeg", "data": b64}}
            ]
        )
        captions.append(response.text.strip())

        if (i + 1) % 10 == 0:
            print(f"  Captioned {i + 1}/{len(images_resized)}")
        time.sleep(0.3)

    np.savez(caption_cache_path, captions=np.array(captions))
    print("Saved captions to Google Drive cache.")

print(f"\nTotal captions: {len(captions)}")
for i in range(min(5, len(captions))):
    print(f"\n  [{i}] {labels[i]}: {captions[i]}")

print("\nEmbedding captions...")
caption_prefixed = ["search_document: " + c for c in captions]
caption_embeddings = text_model.encode(caption_prefixed, show_progress_bar=True)

try:
    chroma_client.delete_collection("caption_index")
except Exception:
    pass

caption_collection = chroma_client.create_collection(name="caption_index")
caption_collection.add(
    ids=[f"cap_{i}" for i in range(len(captions))],
    embeddings=caption_embeddings.tolist(),
    documents=captions,
    metadatas=[{"label": labels[i], "index": i} for i in range(len(captions))]
)

print(f"\nCaption collection: {caption_collection.count()} items, {caption_embeddings.shape[1]}d")
print("You can now search your images by text description.")


In [ ]:
# Experiment 3: Search images by text

def search_by_text(query, n=5):
    q_emb = text_model.encode(["search_query: " + query]).tolist()
    results = caption_collection.query(query_embeddings=q_emb, n_results=n)
    return results

def show_text_search(query, n=5):
    results = search_by_text(query, n=n)

    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
    if n == 1:
        axes = [axes]

    for i, meta in enumerate(results["metadatas"][0]):
        idx = meta["index"]
        dist = results["distances"][0][i]
        axes[i].imshow(images_resized[idx])
        axes[i].set_title(f"{labels[idx]} ({1-dist:.3f})\nidx={idx}", fontsize=9)
        axes[i].axis('off')

    plt.suptitle(f'Query: "{query}"', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"Query: {query}")
    for i, meta in enumerate(results["metadatas"][0]):
        idx = meta["index"]
        print(f"  #{i+1} [{labels[idx]}] (dist={results['distances'][0][i]:.4f}): {captions[idx][:100]}...")

MY_TEXT_QUERY = "a bright yellow flower in a field"
show_text_search(MY_TEXT_QUERY, n=5)


**✍️ Experiment 3 Reflection** (write your answers here)

1. When did the caption bridge work well? What kinds of text queries found the right images?
2. When did it break? Was the failure in Gemini's caption or in the text search?


## Experiment 4: Retrieval Quality Deep Dive

**The question:** How good is this retrieval system? Precision@k tells you: for each query image, of the top k results, how many share the query's class? A perfect system scores 1.0 at every k.

**The procedure:**

1. Run the evaluation cell to compute precision@k across all images.
2. Examine the per-class breakdown. Which classes have the highest precision? Which have the lowest?
3. Find a specific failure case — a query where the model returned the wrong class in the top result.
4. Compare to what you observed in Experiment 1.


In [ ]:
# Experiment 4: Precision@k evaluation

K_VALUES = [1, 3, 5]
from collections import defaultdict

per_class_precision = defaultdict(lambda: {k: [] for k in K_VALUES})
overall_precision = {k: [] for k in K_VALUES}

for i in range(len(labels)):
    query_emb = image_embeddings[i].tolist()
    results = image_collection.query(query_embeddings=[query_emb], n_results=max(K_VALUES) + 1)

    retrieved_labels = []
    for meta in results["metadatas"][0]:
        if meta["index"] != i:
            retrieved_labels.append(meta["label"])

    for k in K_VALUES:
        top_k = retrieved_labels[:k]
        correct = sum(1 for l in top_k if l == labels[i])
        precision = correct / k
        overall_precision[k].append(precision)
        per_class_precision[labels[i]][k].append(precision)

print("=== Overall Precision@k ===")
for k in K_VALUES:
    mean_p = np.mean(overall_precision[k])
    print(f"  Precision@{k}: {mean_p:.3f}")

print(f"\n=== Per-Class Precision@k ===")
print(f"{'Class':<20} {'P@1':>6} {'P@3':>6} {'P@5':>6}")
print("-" * 40)
for cls in sorted(set(labels)):
    row = f"{cls:<20}"
    for k in K_VALUES:
        row += f" {np.mean(per_class_precision[cls][k]):>5.3f}"
    print(row)

worst_class = min(set(labels), key=lambda c: np.mean(per_class_precision[c][1]))
worst_p1 = np.mean(per_class_precision[worst_class][1])
print(f"\nHardest class: {worst_class} (P@1 = {worst_p1:.3f})")

worst_indices = [i for i, l in enumerate(labels) if l == worst_class]
for wi in worst_indices:
    results = search_images_by_image(wi, n=1)
    if results["metadatas"][0][0]["label"] != worst_class:
        print(f"\nFailure case: query idx={wi} ({worst_class})")
        print(f"  Model returned: idx={results['metadatas'][0][0]['index']} ({results['metadatas'][0][0]['label']})")
        show_image_retrieval(wi, n=4)
        break


**✍️ Experiment 4 Reflection** (write your answers here)

1. What was your overall precision@1? What does that number mean in plain English?
2. Which class had the lowest precision? Look at the failure cases. What visual features do those images share with the wrong class?


# Demo Prep

Your notebook is your demo. No separate slides or written reflection needed.

In three to five minutes, walk the class through:

1. **One experiment in depth.** Pick the experiment that surprised you most. Show the code, show the output, and explain what you learned.
2. **A failure.** Show an image where the retrieval system returned the wrong class. Explain what visual feature fooled the model.
3. **The caption bridge.** Show one text query that found the right image and one that didn't. What does this tell you about the gap between visual similarity and language?
4. **One sentence you couldn't have said before this lab.** What do you now understand about embeddings or vector search that you didn't before?

The markdown cells you wrote throughout the lab are your reflection. The demo is your presentation. Come ready to run cells live and talk about what they show.
